# 📊 Notebook 4 — Model Training Validation & Performance Analysis
### Project: Explainable Lightweight Edge-AI Framework for Early Cardiac Risk Prediction
**Week 2 | Task 2: Deep Evaluation, Baseline Comparison & Performance Analysis**

---
**Objectives:**
1. Load trained model from Notebook 3
2. Deep performance analysis (Precision, Recall, F1, AUC)
3. Compare with baseline models (CNN-only, LSTM-only)
4. Per-class performance analysis
5. Error analysis & false prediction study
6. Publication-quality figures
---

## 📦 Cell 1 — Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, Activation,
    MaxPooling1D, Dropout, Bidirectional, LSTM,
    Dense, GlobalAveragePooling1D
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize

# ── Style ─────────────────────────────────────────────────────────
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a2e'
plt.rcParams['axes.edgecolor']   = '#444'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['grid.color']       = '#333'
plt.rcParams['grid.alpha']       = 0.5

os.makedirs('./outputs', exist_ok=True)
os.makedirs('./models', exist_ok=True)

RISK_NAMES  = {0: 'Low Risk', 1: 'Moderate Risk', 2: 'High Risk', 3: 'Critical Risk'}
RISK_COLORS = {0: '#00ff88', 1: '#ffd32a', 2: '#ff6b35', 3: '#ff4757'}
NUM_CLASSES = 4

print('✅ Libraries imported!')
print(f'   TensorFlow : {tf.__version__}')

---
## 📁 Cell 2 — Load Dataset & Trained Model

In [ ]:
# ── Load processed dataset ────────────────────────────────────────
PROCESSED_PATH = r'D:\ECG-Cardiac-Risk-Prediction\notebooks\data\processed'

X = np.load(os.path.join(PROCESSED_PATH, 'X_segments.npy'))
y = np.load(os.path.join(PROCESSED_PATH, 'y_risk_labels.npy'))

# ── Reshape & split (same as Notebook 3) ─────────────────────────
X_dl  = X.reshape(X.shape[0], X.shape[1], 1).astype(np.float32)
y_cat = to_categorical(y, num_classes=NUM_CLASSES)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_dl, y_cat, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42,
    stratify=np.argmax(y_temp, axis=1)
)

y_true = np.argmax(y_test, axis=1)
y_train_int = np.argmax(y_train, axis=1)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_int),
    y=y_train_int
)
class_weight_dict = dict(enumerate(class_weights))

# ── Load trained model ────────────────────────────────────────────
model = keras.models.load_model('./models/ecg_risk_model_final.keras')

print('✅ Dataset & model loaded!')
print(f'   X shape      : {X_dl.shape}')
print(f'   Test samples : {X_test.shape[0]}')
print(f'   Model params : {model.count_params():,}')

---
## 📊 Cell 3 — Full Performance Metrics

In [ ]:
# ── Predictions ───────────────────────────────────────────────────
y_pred_prob  = model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_prob, axis=1)

# ── Core Metrics ──────────────────────────────────────────────────
acc       = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

y_test_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
macro_auc  = roc_auc_score(y_test_bin, y_pred_prob, average='macro', multi_class='ovr')

print('=' * 55)
print('  📊 CNN + BiLSTM + Attention Model — Performance')
print('=' * 55)
print(f'  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision (wtd)   : {precision:.4f}')
print(f'  Recall    (wtd)   : {recall:.4f}')
print(f'  F1-Score  (wtd)   : {f1:.4f}')
print(f'  ROC-AUC   (macro) : {macro_auc:.4f}')
print('=' * 55)

# ── Per-class metrics ─────────────────────────────────────────────
print('\n📋 Per-Class Metrics:')
print(f'  {"Class":<16} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
print('  ' + '-' * 58)
target_names = [RISK_NAMES[i] for i in range(NUM_CLASSES)]
report_dict  = classification_report(
    y_true, y_pred, target_names=target_names,
    output_dict=True, zero_division=0
)
for name in target_names:
    r = report_dict[name]
    print(f'  {name:<16} {r["precision"]:>10.4f} {r["recall"]:>10.4f} '
          f'{r["f1-score"]:>10.4f} {int(r["support"]):>10}')
print('  ' + '-' * 58)

---
## 🏗️ Cell 4 — Build & Train Baseline Models for Comparison

In [ ]:
# ── Baseline 1: CNN Only ──────────────────────────────────────────
def build_cnn_only(input_shape=(180, 1), num_classes=4):
    inputs = Input(shape=input_shape)
    x = Conv1D(32, 5, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)
    x = Conv1D(64, 5, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.2)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs, name='CNN_Only')

# ── Baseline 2: LSTM Only ─────────────────────────────────────────
def build_lstm_only(input_shape=(180, 1), num_classes=4):
    inputs = Input(shape=input_shape)
    x = LSTM(64, return_sequences=True, dropout=0.2)(inputs)
    x = LSTM(32, dropout=0.2)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs, name='LSTM_Only')

# ── Train helper ──────────────────────────────────────────────────
def train_baseline(model, name):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    cb = [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', patience=4, factor=0.5, verbose=0)
    ]
    print(f'🔄 Training {name}...')
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30, batch_size=64,
        class_weight=class_weight_dict,
        callbacks=cb, verbose=0
    )
    y_prob = model.predict(X_test, verbose=0)
    y_p    = np.argmax(y_prob, axis=1)
    acc    = accuracy_score(y_true, y_p)
    f1     = f1_score(y_true, y_p, average='weighted', zero_division=0)
    prec   = precision_score(y_true, y_p, average='weighted', zero_division=0)
    rec    = recall_score(y_true, y_p, average='weighted', zero_division=0)
    auc_s  = roc_auc_score(
        label_binarize(y_true, classes=[0,1,2,3]),
        y_prob, average='macro', multi_class='ovr'
    )
    print(f'   ✅ {name} → Acc={acc:.4f}  F1={f1:.4f}  AUC={auc_s:.4f}')
    return acc, prec, rec, f1, auc_s, history

# ── Build & train baselines ───────────────────────────────────────
cnn_model  = build_cnn_only()
lstm_model = build_lstm_only()

cnn_acc,  cnn_prec,  cnn_rec,  cnn_f1,  cnn_auc,  cnn_hist  = train_baseline(cnn_model,  'CNN Only')
lstm_acc, lstm_prec, lstm_rec, lstm_f1, lstm_auc, lstm_hist = train_baseline(lstm_model, 'LSTM Only')

print('\n✅ All baseline models trained!')

---
## 📊 Cell 5 — Model Comparison Table

In [ ]:
# ── Build comparison dataframe ────────────────────────────────────
comparison_data = {
    'Model'     : ['CNN Only', 'LSTM Only', 'CNN + BiLSTM + Attention (Ours)'],
    'Accuracy'  : [round(cnn_acc,  4), round(lstm_acc,  4), round(acc,       4)],
    'Precision' : [round(cnn_prec, 4), round(lstm_prec, 4), round(precision, 4)],
    'Recall'    : [round(cnn_rec,  4), round(lstm_rec,  4), round(recall,    4)],
    'F1-Score'  : [round(cnn_f1,   4), round(lstm_f1,   4), round(f1,        4)],
    'ROC-AUC'   : [round(cnn_auc,  4), round(lstm_auc,  4), round(macro_auc, 4)],
}
df_compare = pd.DataFrame(comparison_data)

print('📋 Model Comparison Table:')
print('=' * 80)
print(df_compare.to_string(index=False))
print('=' * 80)

df_compare.to_csv('./outputs/model_comparison.csv', index=False)
print('\n✅ Comparison saved → outputs/model_comparison.csv')

---
## 📊 Cell 6 — Model Comparison Bar Chart

In [ ]:
# ── Grouped bar chart comparison ──────────────────────────────────
metrics    = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
models     = ['CNN Only', 'LSTM Only', 'CNN+BiLSTM+Attn']
cnn_vals   = [cnn_acc,  cnn_prec,  cnn_rec,  cnn_f1,  cnn_auc]
lstm_vals  = [lstm_acc, lstm_prec, lstm_rec, lstm_f1, lstm_auc]
our_vals   = [acc,      precision, recall,   f1,      macro_auc]

x     = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))

bars1 = ax.bar(x - width,     cnn_vals,  width, label='CNN Only',          color='#45b7d1', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x,             lstm_vals, width, label='LSTM Only',         color='#96ceb4', edgecolor='white', linewidth=0.5)
bars3 = ax.bar(x + width,     our_vals,  width, label='CNN+BiLSTM+Attn ⭐', color='#ffd32a', edgecolor='white', linewidth=0.5)

# Value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8, color='white')

ax.set_title('Model Comparison — ECG Cardiac Risk Prediction', fontsize=13, color='white')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim([0, 1.12])
ax.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=10)
ax.grid(axis='y')

plt.tight_layout()
plt.savefig('./outputs/fig14_model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig14_model_comparison.png')

---
## 📊 Cell 7 — Training History Comparison

In [ ]:
# ── Load Notebook 3 training history ─────────────────────────────
history_df = pd.read_csv('./outputs/training_history.csv')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy comparison
axes[0].plot(cnn_hist.history['val_accuracy'],
             color='#45b7d1', linewidth=1.5, linestyle='--', label='CNN Only (Val)')
axes[0].plot(lstm_hist.history['val_accuracy'],
             color='#96ceb4', linewidth=1.5, linestyle='--', label='LSTM Only (Val)')
axes[0].plot(history_df['val_accuracy'],
             color='#ffd32a', linewidth=2.5, label='CNN+BiLSTM+Attn (Val) ⭐')
axes[0].set_title('Validation Accuracy Comparison', fontsize=12, color='white')
axes[0].set_xlabel('Epoch', fontsize=10)
axes[0].set_ylabel('Accuracy', fontsize=10)
axes[0].legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=9)
axes[0].grid(True)

# Loss comparison
axes[1].plot(cnn_hist.history['val_loss'],
             color='#45b7d1', linewidth=1.5, linestyle='--', label='CNN Only (Val)')
axes[1].plot(lstm_hist.history['val_loss'],
             color='#96ceb4', linewidth=1.5, linestyle='--', label='LSTM Only (Val)')
axes[1].plot(history_df['val_loss'],
             color='#ffd32a', linewidth=2.5, label='CNN+BiLSTM+Attn (Val) ⭐')
axes[1].set_title('Validation Loss Comparison', fontsize=12, color='white')
axes[1].set_xlabel('Epoch', fontsize=10)
axes[1].set_ylabel('Loss', fontsize=10)
axes[1].legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=9)
axes[1].grid(True)

fig.suptitle('Training Convergence Comparison — All Models', fontsize=13, color='white')
plt.tight_layout()
plt.savefig('./outputs/fig15_training_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig15_training_comparison.png')

---
## 📊 Cell 8 — Per-Class Performance Radar Chart

In [ ]:
# ── Per-class F1 scores ───────────────────────────────────────────
f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
pr_per_class = precision_score(y_true, y_pred, average=None, zero_division=0)
rc_per_class = recall_score(y_true, y_pred, average=None, zero_division=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x_pos      = np.arange(NUM_CLASSES)
bar_colors = [RISK_COLORS[i] for i in range(NUM_CLASSES)]
class_labels = [RISK_NAMES[i] for i in range(NUM_CLASSES)]

for ax, values, title in zip(
    axes,
    [pr_per_class, rc_per_class, f1_per_class],
    ['Precision per Class', 'Recall per Class', 'F1-Score per Class']
):
    bars = ax.bar(x_pos, values, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.set_title(title, fontsize=11, color='white')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(class_labels, rotation=15, fontsize=9)
    ax.set_ylabel('Score', fontsize=10)
    ax.set_ylim([0, 1.15])
    ax.grid(axis='y')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=9, color='white')

fig.suptitle('Per-Class Performance — CNN + BiLSTM + Attention Model',
             fontsize=13, color='white')
plt.tight_layout()
plt.savefig('./outputs/fig16_per_class_performance.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig16_per_class_performance.png')

---
## 🔍 Cell 9 — Error Analysis & False Predictions

In [ ]:
# ── Find misclassified samples ────────────────────────────────────
wrong_idx      = np.where(y_pred != y_true)[0]
correct_idx    = np.where(y_pred == y_true)[0]

print(f'🔍 Error Analysis:')
print(f'   Total test samples  : {len(y_true)}')
print(f'   Correctly classified: {len(correct_idx)}  ({100*len(correct_idx)/len(y_true):.2f}%)')
print(f'   Misclassified       : {len(wrong_idx)}   ({100*len(wrong_idx)/len(y_true):.2f}%)')

# ── Confusion pairs (what gets confused with what) ────────────────
print('\n❌ Most Common Misclassification Pairs:')
confusion_pairs = {}
for idx in wrong_idx:
    pair = (RISK_NAMES[y_true[idx]], RISK_NAMES[y_pred[idx]])
    confusion_pairs[pair] = confusion_pairs.get(pair, 0) + 1

sorted_pairs = sorted(confusion_pairs.items(), key=lambda x: -x[1])
print(f'   {"True Label":<16} → {"Predicted":<16} {"Count"}')
print('   ' + '-' * 45)
for (true_l, pred_l), count in sorted_pairs[:8]:
    print(f'   {true_l:<16} → {pred_l:<16} {count}')

# ── Confidence analysis ───────────────────────────────────────────
correct_conf = y_pred_prob[correct_idx, y_pred[correct_idx]]
wrong_conf   = y_pred_prob[wrong_idx,   y_pred[wrong_idx]]

print(f'\n📊 Confidence Analysis:')
print(f'   Correct predictions → Mean confidence : {correct_conf.mean():.4f}')
print(f'   Wrong predictions   → Mean confidence : {wrong_conf.mean():.4f}')

---
## 📊 Cell 10 — Confidence Distribution Plot

In [ ]:
# ── Confidence distribution ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of prediction confidence
axes[0].hist(correct_conf, bins=30, color='#00ff88', alpha=0.7,
             label=f'Correct (n={len(correct_conf)})', edgecolor='white', linewidth=0.3)
axes[0].hist(wrong_conf, bins=30, color='#ff4757', alpha=0.7,
             label=f'Wrong (n={len(wrong_conf)})', edgecolor='white', linewidth=0.3)
axes[0].set_title('Prediction Confidence Distribution', fontsize=12, color='white')
axes[0].set_xlabel('Confidence Score', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
axes[0].grid(True)

# Per-class accuracy bar
per_class_acc = []
for c in range(NUM_CLASSES):
    idx_c = np.where(y_true == c)[0]
    if len(idx_c) > 0:
        per_class_acc.append(np.mean(y_pred[idx_c] == c))
    else:
        per_class_acc.append(0)

bars = axes[1].bar(
    [RISK_NAMES[i] for i in range(NUM_CLASSES)],
    per_class_acc,
    color=[RISK_COLORS[i] for i in range(NUM_CLASSES)],
    edgecolor='white', linewidth=0.5
)
axes[1].set_title('Per-Class Accuracy', fontsize=12, color='white')
axes[1].set_ylabel('Accuracy', fontsize=10)
axes[1].set_ylim([0, 1.15])
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y')
for bar, val in zip(bars, per_class_acc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontsize=10, color='white')

fig.suptitle('Error & Confidence Analysis', fontsize=13, color='white')
plt.tight_layout()
plt.savefig('./outputs/fig17_confidence_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig17_confidence_analysis.png')

---
## 📊 Cell 11 — Final Summary Report

In [ ]:
# ── Save final comparison ─────────────────────────────────────────
df_compare.to_csv('./outputs/model_comparison.csv', index=False)

print('=' * 60)
print('  🎉 WEEK 2 COMPLETE — Both Notebooks Done!')
print('=' * 60)
print()
print('📊 Final Model Performance (CNN + BiLSTM + Attention):')
print(f'   Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'   Precision : {precision:.4f}')
print(f'   Recall    : {recall:.4f}')
print(f'   F1-Score  : {f1:.4f}')
print(f'   ROC-AUC   : {macro_auc:.4f}')
print()
print('📁 Figures Generated:')
for i, name in enumerate([
    'fig14_model_comparison',
    'fig15_training_comparison',
    'fig16_per_class_performance',
    'fig17_confidence_analysis'
], start=14):
    print(f'   ✅ {name}.png')
print()
print('➡️  Next: Week 3 — Notebook 5 (Explainable AI & Attention Visualization)')
print('=' * 60)

---
## ✅ Notebook 4 Summary

| Task | Status |
|------|--------|
| Model loaded from Notebook 3 | ✅ Done |
| Full metrics (Acc/Prec/Rec/F1/AUC) | ✅ Done |
| CNN-Only baseline trained | ✅ Done |
| LSTM-Only baseline trained | ✅ Done |
| Model comparison table | ✅ Done |
| Grouped bar chart comparison | ✅ Done |
| Training convergence comparison | ✅ Done |
| Per-class performance analysis | ✅ Done |
| Error & false prediction analysis | ✅ Done |
| Confidence distribution plot | ✅ Done |

**Week 2 Complete! Ready for Week 3 → Explainable AI**